# Run the full benchmark on Colab (GPU)

Steps: GPU on → get code → install → data → keys → run → report → download.

**One-time setup:** in Colab's 🔑 *Secrets* panel (left sidebar) add TWO secrets and
switch on "Notebook access" for both:
- `HF_TOKEN`: your Hugging Face read token (Hugging Face hosts TabPFN's model files)
- `TABPFN_TOKEN`: your Prior Labs API key (from https://platform.priorlabs.ai/account/api-keys),
  which proves you accepted TabPFN's licence

Keys are read from Secrets, never typed into the notebook, so they can't leak via GitHub.

## 1. Check the GPU is on
Runtime → Change runtime type → **T4 GPU** → Save. Colab resets this when a session ends, so check it every time. This cell should print a GPU name; if it errors, the GPU is off.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code from GitHub
Replace the URL with your repo's URL.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/tabpfn-small-noisy-benchmark.git"  # <-- edit
!git clone -q {REPO_URL} project
%cd project

## 3. Install packages
Uses the same pinned versions as the laptop.

In [ ]:
!pip install -q -r requirements.txt

## 4. Add the data files
Upload the four files from your laptop's `data/` folder:
`winequality-red.csv`, `winequality-white.csv`, `adult.data`, `adult.test`.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()          # opens a file picker
for name in uploaded:
    shutil.move(name, f"data/{name}")
!ls data

## 5. Give TabPFN both keys (read from Colab Secrets)

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # Hugging Face: download files
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")  # Prior Labs: licence check
os.environ["TABPFN_NO_BROWSER"] = "1"   # skip browser login (it can't work in Colab)
print("Keys loaded:", bool(os.environ["HF_TOKEN"]), bool(os.environ["TABPFN_TOKEN"]))

Quick TabPFN check: should print a GPU name and a probability table, not a licence error.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn the GPU on")
from src.data import load_dataset
from src.models import make_model
ds = load_dataset("wine")
m = make_model("tabpfn", ds.numeric, ds.categorical, seed=0)
m.fit(ds.X[:100], ds.y[:100])
print(m.predict_proba(ds.X[100:103]).round(3))

## 6. Save results to Google Drive (survives disconnects)

Results are written to Drive after every model fit. If Colab disconnects, re-run
cells 1–6 and then cell 7: finished work is skipped automatically.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/small-tabular-benchmark/raw_results.csv"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

## 7. Quick check first (a few minutes), then the full run

In [ ]:
!python -m src.experiment --quick --out /content/drive/MyDrive/small-tabular-benchmark/raw_results_quick.csv

In [ ]:
!python -m src.experiment --out {OUT}

## 8. Make the summary tables and plots

In [ ]:
!python -m src.report --raw {OUT}
!ls results/tables results/figures

## 9. Download results
Unzip into your laptop's project folder, then commit them with git.

In [ ]:
!cp {OUT} results/tables/raw_results.csv
!zip -qr results.zip results
files.download("results.zip")